In [ ]:
import pandas as pd
# phase2 
df1 = pd.read_csv('/Users/qwergs/Desktop/필드/phase2/비트코인.csv')
df2 = pd.read_csv('/Users/qwergs/Desktop/필드/phase2/이더리움.csv')
df3 = pd.read_csv('/Users/qwergs/Desktop/필드/phase2/채권.csv')
# etf = pd.read_csv('/Users/qwergs/Desktop/필드/phase2/국내ETF.csv', encoding = enc)
# kospi_tpo100 = pd.read_csv('/Users/qwergs/Desktop/필드/phase2/KOSPI_Top100.csv')

- 파일 인코딩 자동 탐지

In [ ]:
import chardet

# 파일 앞부분만 읽어서 인코딩 추측
raw = open('/Users/qwergs/Desktop/필드/phase2/국내ETF.csv', 'rb').read(10000)
enc = chardet.detect(raw)['encoding']
print("추측된 인코딩:", enc)

# 추측된 인코딩으로 다시 읽기
df4 = pd.read_csv(
    '/Users/qwergs/Desktop/필드/phase2/국내ETF.csv',
    encoding=enc
)

In [ ]:
import chardet

# 파일 앞부분 바이트를 읽어서 인코딩 추측
raw = open('/Users/qwergs/Desktop/필드/phase2/KOSPI_Top100.csv','rb').read(5000)
enc = chardet.detect(raw)['encoding']
print("추측 인코딩:", enc)

# 추측된 인코딩으로 다시 읽기
df5 = pd.read_csv(
    '/Users/qwergs/Desktop/필드/phase2/KOSPI_Top100.csv',
    encoding=enc
)

In [ ]:
btc = df1.copy()
edh = df2.copy()
bond = df3.copy()
etf = df4.copy()
kospi = df5.copy()

In [ ]:
print(bond.index.min(), bond.index.max())

# 클러스터 0, 기본 합리형
- 변동성 최소화 
- 투자 성향 : 보수적 / 안정지향
- 최소 분산 포트폴리오 -> 

In [ ]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize

# 1. 데이터 정제
bond_cols = ['1 Mo', '1.5 Month', '2 Mo', '3 Mo', '4 Mo', '6 Mo', '1 Yr',
             '2 Yr', '3 Yr', '5 Yr', '7 Yr', '10 Yr', '20 Yr', '30 Yr']

# 날짜 변환
df1['Date'] = pd.to_datetime(df1['Date'])  # BTC
df2['Date'] = pd.to_datetime(df2['Date'])  # ETH
df3['Date'] = pd.to_datetime(df3['Date'], format='%m/%d/%Y')  # Bonds
df4['Date'] = pd.to_datetime(df4['Date'])  # ETF
df5['Date'] = pd.to_datetime(df5['Date'])  # KOSPI

# 가격 데이터셋
btc = df1[['Date', 'Close']].rename(columns={'Close': 'BTC'}).set_index('Date')
eth = df2[['Date', 'Close']].rename(columns={'Close': 'ETH'}).set_index('Date')
etf = df4.pivot(index='Date', columns='ETF_Name', values='Close')
kospi = df5.pivot(index='Date', columns='Name', values='Close')
bond = df3[['Date'] + bond_cols].set_index('Date')

# 2. 수익률 계산
# 자산 수익률 (3년 전체)
price_df = btc.join(eth, how='outer') \
              .join(etf, how='outer') \
              .join(kospi, how='outer')
returns_asset = price_df.pct_change().dropna(thresh=int(0.8 * price_df.shape[1]))
# returns_asset = price_df.pct_change(fill_method=None).dropna(thresh=int(0.8 * price_df.shape[1]))
returns_asset = returns_asset.loc[:, returns_asset.std() > 0.01]

# 채권 수익률 (최근 6개월)
bond = bond.sort_index()
bond_recent = bond.loc['2025-01-02':'2025-06-27']
returns_bond = bond_recent.pct_change().dropna(thresh=int(0.8 * len(bond_cols)))

# 3. 채권 포트폴리오 고정 (가중치 조정 가능)
bond_weight = 0.25
bond_mean = returns_bond.mean()
bond_cov = returns_bond.cov()

# 균등 분산 투자
num_bonds = len(bond_cols)
bond_equal_weights = np.ones(num_bonds) / num_bonds
bond_equal_weights *= bond_weight  # 고정 비중 반영

# 4. 자산 포트폴리오 최적화 
target_assets = returns_asset.columns.tolist()
asset_cov = returns_asset.cov()

def objective(weights, cov_matrix):
    full_weights = np.append(weights, bond_equal_weights)
    full_cov = cov_matrix
    vol = np.sqrt(np.dot(full_weights.T, np.dot(full_cov, full_weights)))
    penalty = 10000 * max(0, 5 - np.sum(weights >= 0.02))  # 다양성 유도
    return vol + penalty

# 전체 공분산 행렬 (자산 + 채권)
combined_returns = pd.concat([returns_asset, returns_bond], axis=1)
combined_cov = combined_returns.cov()

init_weights = np.ones(len(target_assets)) / len(target_assets)
bounds = [(0, bond_weight)] * len(target_assets)
constraints = {'type': 'eq', 'fun': lambda x: np.sum(x) - (1 - bond_weight)}

result = minimize(objective,
                  init_weights,
                  args=(combined_cov,),
                  method='SLSQP',
                  bounds=bounds,
                  constraints=constraints)

# 5. 결과 정리
final_weights = np.append(result.x, bond_equal_weights)
final_assets = target_assets + bond_cols

allocation = pd.DataFrame({'Asset': final_assets, 'Weight': final_weights})
allocation = allocation[allocation['Weight'] > 0.005].sort_values(by='Weight', ascending=False)

print(allocation)

In [ ]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize

# 1. 데이터 정제
bond_cols = ['1 Mo', '1.5 Month', '2 Mo', '3 Mo', '4 Mo', '6 Mo', '1 Yr',
             '2 Yr', '3 Yr', '5 Yr', '7 Yr', '10 Yr', '20 Yr', '30 Yr']

# 날짜 변환
df1['Date'] = pd.to_datetime(df1['Date'])  # BTC
df2['Date'] = pd.to_datetime(df2['Date'])  # ETH
df3['Date'] = pd.to_datetime(df3['Date'], format='%m/%d/%Y')  # Bonds
df4['Date'] = pd.to_datetime(df4['Date'])  # ETF
df5['Date'] = pd.to_datetime(df5['Date'])  # KOSPI

# 가격 데이터셋
btc = df1[['Date', 'Close']].rename(columns={'Close': 'BTC'}).set_index('Date')
eth = df2[['Date', 'Close']].rename(columns={'Close': 'ETH'}).set_index('Date')
etf = df4.pivot(index='Date', columns='ETF_Name', values='Close')
kospi = df5.pivot(index='Date', columns='Name', values='Close')
bond = df3[['Date'] + bond_cols].set_index('Date')

# 2. 수익률 계산
# 자산 수익률 (3년 전체)
price_df = btc.join(eth, how='outer') \
              .join(etf, how='outer') \
              .join(kospi, how='outer')
returns_asset = price_df.pct_change().dropna(thresh=int(0.8 * price_df.shape[1]))
returns_asset = returns_asset.loc[:, returns_asset.std() > 0.01]

# 채권 수익률 (최근 6개월)
bond = bond.sort_index()
bond_recent = bond.loc['2025-01-02':'2025-06-27']
returns_bond = bond_recent.pct_change().dropna(thresh=int(0.8 * len(bond_cols)))

# 3. 채권 포트폴리오 고정 (비중 0.2)
bond_weight = 0.2
bond_mean = returns_bond.mean()
bond_cov = returns_bond.cov()

# 균등 분산 투자
num_bonds = len(bond_cols)
bond_equal_weights = np.ones(num_bonds) / num_bonds
bond_equal_weights *= bond_weight  # 고정 비중 반영

# 4. 자산 포트폴리오 최적화
target_assets = returns_asset.columns.tolist()
asset_cov = returns_asset.cov()

def objective(weights, cov_matrix):
    full_weights = np.append(weights, bond_equal_weights)
    full_cov = cov_matrix
    vol = np.sqrt(np.dot(full_weights.T, np.dot(full_cov, full_weights)))
    penalty = 10000 * max(0, 5 - np.sum(weights >= 0.01))  # 다양성 유도
    return vol + penalty

# 전체 공분산 행렬 (자산 + 채권)
combined_returns = pd.concat([returns_asset, returns_bond], axis=1)
combined_cov = combined_returns.cov()

init_weights = np.ones(len(target_assets)) / len(target_assets)
bounds = [(0, bond_weight)] * len(target_assets)
constraints = {'type': 'eq', 'fun': lambda x: np.sum(x) - (1 - bond_weight)}

result = minimize(objective,
                  init_weights,
                  args=(combined_cov,),
                  method='SLSQP',
                  bounds=bounds,
                  constraints=constraints)

# 5. 결과 정리
final_weights = np.append(result.x, bond_equal_weights)
final_assets = target_assets + bond_cols

allocation = pd.DataFrame({'Asset': final_assets, 'Weight': final_weights})
allocation = allocation[allocation['Weight'] > 0.015].sort_values(by='Weight', ascending=False)

print(allocation)

# 클러스터 1 , 효율 중심 자립형 
- 자산선택 기준 : 수익형 
- 투자 성향 : 자산 풍부, 계획형
- CVaR 최소화를 통한 tail risk 회피 전략

In [ ]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize

# 1. 데이터 전처리
bond_cols = ['1 Mo', '1.5 Month', '2 Mo', '3 Mo', '4 Mo', '6 Mo', '1 Yr',
             '2 Yr', '3 Yr', '5 Yr', '7 Yr', '10 Yr', '20 Yr', '30 Yr']

# 날짜 변환
df1['Date'] = pd.to_datetime(df1['Date'])  # BTC
df2['Date'] = pd.to_datetime(df2['Date'])  # ETH
df3['Date'] = pd.to_datetime(df3['Date'], format='%m/%d/%Y')  # Bonds
df4['Date'] = pd.to_datetime(df4['Date'])  # ETF
df5['Date'] = pd.to_datetime(df5['Date'])  # KOSPI

# 가격 병합
btc = df1[['Date', 'Close']].rename(columns={'Close': 'BTC'}).set_index('Date')
eth = df2[['Date', 'Close']].rename(columns={'Close': 'ETH'}).set_index('Date')
etf = df4.pivot(index='Date', columns='ETF_Name', values='Close')
kospi = df5.pivot(index='Date', columns='Name', values='Close')
bond = df3[['Date'] + bond_cols].set_index('Date')

# 수익률 계산
price_df = btc.join(eth, how='outer') \
              .join(etf, how='outer') \
              .join(kospi, how='outer')

returns_asset = price_df.pct_change(fill_method=None).dropna()
returns_asset = returns_asset.loc[:, returns_asset.std(skipna=True) > 0.0]

# 채권 수익률 (6개월)
bond = bond.sort_index()
bond_recent = bond.loc['2025-01-02':'2025-06-27']
returns_bond = bond_recent.pct_change().dropna(thresh=int(0.8 * len(bond_cols)))

# 2. 고정 채권 비중
bond_weight = 0.25
num_bonds = len(bond_cols)
bond_equal_weights = np.ones(num_bonds) / num_bonds
bond_equal_weights *= bond_weight

# 채권 가중치 설정
bond_weight = 0.20
num_bonds = len(bond_cols)
bond_equal_weights = np.ones(num_bonds) / num_bonds
bond_equal_weights *= bond_weight


# 모든 자산 수익률 통합
price_df = btc.join(eth, how='outer').join(etf, how='outer').join(kospi, how='outer')
returns_asset = price_df.pct_change().dropna(how='all')
returns_asset = returns_asset.loc[:, returns_asset.std(skipna=True) > 0.0]

bond = bond.sort_index()
bond_recent = bond.loc['2025-01-02':'2025-06-27']
returns_bond = bond_recent.pct_change().dropna(thresh=int(0.8 * len(bond_cols)))

returns_all = pd.concat([returns_asset, returns_bond], axis=1).dropna()
asset_cols = returns_all.columns.tolist()
n_assets = len(asset_cols)

# CVaR 계산 + 선택 개수 제약 패널티
def penalty_asset_count(weights, min_assets=5, max_assets=10, threshold=0.005, penalty_scale=100):
    count = np.sum(weights > threshold)
    if count < min_assets:
        return penalty_scale * (min_assets - count)**2
    elif count > max_assets:
        return penalty_scale * (count - max_assets)**2
    return 0.0

def portfolio_cvar_with_penalty(weights, return_data, alpha=0.95):
    port_returns = return_data @ weights
    port_returns = port_returns.dropna()
    var = np.percentile(port_returns, (1 - alpha) * 100)
    cvar = port_returns[port_returns <= var].mean()

    penalty = penalty_asset_count(weights)
    return -cvar + penalty

# 최적화 설정
init_weights = np.ones(n_assets) / n_assets
bounds = [(0, 1)] * n_assets
constraints = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}

# 최적화 수행
result = minimize(portfolio_cvar_with_penalty,
                  init_weights,
                  args=(returns_all,),
                  method='SLSQP',
                  bounds=bounds,
                  constraints=constraints)

# 결과 정리
weights_final = result.x
allocation = pd.DataFrame({
    'Asset': asset_cols,
    'Weight': weights_final
})
allocation = allocation[allocation['Weight'] > 0.005].sort_values(by='Weight', ascending=False)

print(allocation)

### `cVar 고려사항`
- Alpha값 튜닝 : 일반적으로는 0.95, 0.99룰 사용, 클러스터의 리스크 성향에 따라 조정 가능 
- 수익률 필터링 : 극단적 이상치 제거 또는 winsorize 처리로 안정적인 tail risk 추정 가능 
- ETF와 채권 중심으로 구성 가능 : 고자산층은 변동성이 낮은 안정적 자산에 선호를 둘 수 있음 -> 특정 자산군으로 제한적 cVar 적용도 가능 

# 클러스터 2, 수익 극대화 

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize

# 수익률 계산 (모든 자산 통합: BTC, ETH, ETF, KOSPI, Bonds)
# 모든 자산 통합
price_df = btc.join(edh, how='outer').join(etf, how='outer').join(kospi, how='outer')
# 숫자형만 필터
price_df = price_df.select_dtypes(include='number')
# 결측치 처리 : 선형 보간 (가로축: 날짜 방향)
price_df = price_df.interpolate(method='linear', axis=0, limit_direction='both')
# 수익률 계산
returns_asset = price_df.pct_change(fill_method=None).dropna()

bond_recent = bond.loc['2025-01-02':'2025-06-27']
bond_recent = bond_recent.select_dtypes(include='number')
bond_recent = bond_recent.interpolate(method='linear', axis=0, limit_direction='both')
returns_bond = bond_recent.pct_change(fill_method=None).dropna()

returns_all = pd.concat([returns_asset, returns_bond], axis=1).dropna()
asset_cols = returns_all.columns.tolist()
n_assets = len(asset_cols)

# 기대 수익률 & 공분산 계산
mu = returns_all.mean()  # 기대 수익률 (평균)
cov_matrix = returns_all.cov()  # 공분산

# 기대 수익률 함수 (부호 반대로: maximize → minimize)
def negative_portfolio_return(weights, mu):
    return -np.dot(weights, mu)

# 선택 자산 수 제약 패널티 함수
def penalty_asset_count(weights, min_assets=6, max_assets=10, threshold=0.005, penalty_scale=100):
    count = np.sum(weights > threshold)
    if count < min_assets:
        return penalty_scale * (min_assets - count)**2
    elif count > max_assets:
        return penalty_scale * (count - max_assets)**2
    return 0.0

# 최종 목적 함수
def objective_function(weights, mu, cov_matrix, risk_aversion=0.05):
    ret = np.dot(weights, mu)
    var = np.dot(weights.T, np.dot(cov_matrix, weights))
    penalty = penalty_asset_count(weights)
    return -ret + risk_aversion * var + penalty

# 초기값, 제약, 경계
init_weights = np.ones(n_assets) / n_assets
bounds = [(0, 1)] * n_assets
constraints = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}

# 최적화 수행
result = minimize(objective_function,
                  init_weights,
                  args=(mu, cov_matrix),
                  method='SLSQP',
                  bounds=bounds,
                  constraints=constraints)

# 결과 정리
weights_final = result.x
allocation = pd.DataFrame({
    'Asset': asset_cols,
    'Weight': weights_final
})
allocation = allocation[allocation['Weight'] > 0.005].sort_values(by='Weight', ascending=False)

print(allocation)

` Markowitz 평균-분산 최적화 고려사항`
- risk_aversion = 0.05 는 수익/위험 균형을 조절하는 하이퍼파라미터
    - 위험 회피 성향을 수치화한 것
    - 낮을수록 수익 극대화에 집중 
    - 1 이상이면 위험을 매우 민감하게 회피 -> 보수적 
- penalty_asset_count() 함수로 선택된 자산 수 5~10개 유지

# 클러스터 3, 고립된 회피형 
- 위험 아예 배제
- 보수적 
- Markowitz 평균-분산 최적화 모델 사용 

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize

# 1. 데이터 결합 및 전처리
price_df = btc.join(edh, how='outer').join(etf, how='outer').join(kospi, how='outer')
price_df = price_df.interpolate(method='linear')  # 선형 보간
price_df = price_df.select_dtypes(include=[np.number])
returns_asset = price_df.pct_change().dropna(how='all')
returns_asset = returns_asset.loc[:, returns_asset.std(skipna=True) > 0]

bond_recent = bond.loc['2025-01-02':'2025-06-27'].interpolate(method='linear')
bond_recent = bond.loc['2025-01-02':'2025-06-27'].select_dtypes(include=[np.number])
returns_bond = bond_recent.pct_change().dropna(thresh=int(0.8 * len(bond_recent.columns)))

returns_all = pd.concat([returns_asset, returns_bond], axis=1).dropna()
asset_cols = returns_all.columns.tolist()
n_assets = len(asset_cols)

# 2. 평균 수익률 및 공분산 행렬 계산
mu = returns_all.mean()
cov = returns_all.cov()

# 3. 위험 회피 계수 설정
risk_aversion = 10  # 높을수록 리스크를 더 꺼림 (기본: 0.05 ~ 10 사이 조정 가능)

# 4. 목적 함수 (expected utility = mean - risk_aversion * variance)
def objective(w, mu, cov, risk_aversion):
    expected_return = np.dot(w, mu)
    risk = np.dot(w.T, np.dot(cov, w))
    return - (expected_return - risk_aversion * risk)

# 5. 선택 자산 개수 제약 함수 (5~10개 종목 유지)
def penalty_asset_count(weights, min_assets=5, max_assets=10, threshold=0.005, penalty_scale=0):
    count = np.sum(weights > threshold)
    if count < min_assets:
        return penalty_scale * (min_assets - count)**2
    elif count > max_assets:
        return penalty_scale * (count - max_assets)**2
    return 0

# 6. 전체 목적 함수 (효용 - 패널티)
def full_objective(w, mu, cov, risk_aversion):
    return objective(w, mu, cov, risk_aversion) + penalty_asset_count(w)

# 7. 최적화 설정
init_weights = np.ones(n_assets) / n_assets
bounds = [(0, 1)] * n_assets
constraints = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}

# 8. 최적화 실행
res = minimize(full_objective,
               init_weights,
               args=(mu, cov, risk_aversion),
               method='SLSQP',
               bounds=bounds,
               constraints=constraints)

# 9. 결과 정리
w = res.x
allocation = pd.DataFrame({'Asset': asset_cols, 'Weight': w})
allocation = allocation[allocation['Weight'] > 0.005].sort_values(by='Weight', ascending=False)

# 10. 출력
print(f"[클러스터 3 - Markowitz 모델 결과 | 위험 회피 계수 = {risk_aversion}]")
print(allocation)

# 클러스터 4. 자기 방어형 고소득 
- 채권 중심, 보수적 ETF 
- Risk Parity Portfolio 사용 

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize

for df in [btc, edh, etf, kospi]:
    df.index = pd.to_datetime(df.index, errors='coerce')
bond.index = pd.to_datetime(bond.index, errors='coerce')

# 1. 자산 수익률 계산
# price_df = btc.join(edh, how='outer').join(etf, how='outer').join(kospi, how='outer')
# price_df = price_df.interpolate(method='linear')
# price_df = price_df.select_dtypes(include=[np.number])
price_df = btc.join(edh, how='outer').join(etf, how='outer').join(kospi, how='outer')
price_df = price_df.infer_objects(copy=False)  # object → float 변환
price_df = price_df.interpolate(method='linear', axis=0)
price_df = price_df.select_dtypes(include=[np.number])  # 수치형만 유지

# returns_asset = price_df.pct_change().dropna(how='all')
# returns_asset = returns_asset.loc[:, returns_asset.std(skipna=True) > 0]
returns_asset = price_df.pct_change().dropna(how='all')
returns_asset = returns_asset.loc[:, returns_asset.std(skipna=True) > 0]

bond_recent = bond.loc['2025-01-02':'2025-06-27'].interpolate(method='linear')
bond_recent = bond_recent.select_dtypes(include=[np.number])
returns_bond = bond_recent.pct_change().dropna(thresh=int(0.8 * len(bond_recent.columns)))

returns_all = pd.concat([returns_asset, returns_bond], axis=1).dropna()
asset_cols = returns_all.columns.tolist()
n_assets = len(asset_cols)
volatility = returns_all.std()

# 2. Risk Parity 목적 함수 + 자산 개수 제약 포함
def risk_parity_objective_with_penalty(weights, cov, penalty_lambda=1000):
    portfolio_vol = np.sqrt(np.dot(weights.T, np.dot(cov, weights)))
    marginal_risk_contrib = np.dot(cov, weights) / portfolio_vol
    risk_contrib = weights * marginal_risk_contrib
    risk_contrib_pct = risk_contrib / portfolio_vol
    rp_loss = np.sum((risk_contrib_pct - 1/n_assets)**2)

    # 자산 개수 제약 페널티 추가 (5~10개 이외는 패널티)
    active_assets = np.sum(weights > 0.005)  # 선택된 자산 수
    if active_assets < 5 or active_assets > 10:
        penalty = penalty_lambda * (min(abs(active_assets - 5), abs(active_assets - 10)))
    else:
        penalty = 0

    return rp_loss + penalty

# 3. 채권 최소 가중치 제약 함수 (옵션 적용)
def bond_min_constraint(weights, asset_names, bond_min_ratio=0.2):
    bond_idx = [i for i, name in enumerate(asset_names) if 'Yr' in name or 'Bond' in name]
    bond_weight = np.sum([weights[i] for i in bond_idx])
    return bond_weight - bond_min_ratio

# 4. 최적화 실행
cov = returns_all.cov()
init_weights = np.ones(n_assets) / n_assets
bounds = [(0, 1)] * n_assets
constraints = [
    {'type': 'eq', 'fun': lambda w: np.sum(w) - 1},  # 전체 가중치 합 = 1
    {'type': 'ineq', 'fun': bond_min_constraint, 'args': (asset_cols, 0.2)}  # 채권 최소 20%
]

result = minimize(
    risk_parity_objective_with_penalty,
    init_weights,
    args=(cov, 1000),  # penalty_lambda = 1000
    method='SLSQP',
    bounds=bounds,
    constraints=constraints
)

# 5. 결과 정리
w = result.x
allocation = pd.DataFrame({'Asset': asset_cols, 'Weight': w})
allocation = allocation[allocation['Weight'] > 0.005].sort_values(by='Weight', ascending=False)

# 6. 출력
print("[클러스터 4 - Risk Parity 모델 결과 | 채권 최소 비중 20% 적용]")
print(allocation)

# 클러스터 5. 지원의존형 
- 초보자용 분산, 방어적 투자 
- 단순 분산 투자, 안정성 우선 
- Minimum Variance Portfolio (MVP) 사용 

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize

# 1. 날짜 정리
for df in [btc, edh, etf, kospi]:
    df.index = pd.to_datetime(df.index, errors='coerce')
bond.index = pd.to_datetime(bond.index, errors='coerce')

# 2. 가격 데이터 정리 및 수익률 계산
price_df = btc.join(edh, how='outer').join(etf, how='outer').join(kospi, how='outer')
price_df = price_df.infer_objects(copy=False)
price_df = price_df.interpolate(method='linear', axis=0)
price_df = price_df.select_dtypes(include=[np.number])

returns_asset = price_df.pct_change().dropna(how='all')
returns_asset = returns_asset.loc[:, returns_asset.std(skipna=True) > 0]

bond_recent = bond.loc['2025-01-02':'2025-06-27'].interpolate(method='linear')
bond_recent = bond_recent.select_dtypes(include=[np.number])
returns_bond = bond_recent.pct_change().dropna(thresh=int(0.8 * len(bond_recent.columns)))

returns_all = pd.concat([returns_asset, returns_bond], axis=1).dropna()
asset_cols = returns_all.columns.tolist()
n_assets = len(asset_cols)

# 3. Soft penalty를 포함한 MVP 목적 함수 정의
# def objective_with_soft_penalty(weights, cov, penalty=1000):
#     var = np.dot(weights.T, np.dot(cov, weights))
#     num_selected = np.sum(weights > 0.005)
#     if 5 <= num_selected <= 10:
#         return var
#     else:
#         penalty_term = penalty * (abs(num_selected - 7.5))  # 중심을 7.5로 두고 패널티
#         return var + penalty_term

# 4. 최적화 실행
# cov = returns_all.cov()
# init_weights = np.ones(n_assets) / n_assets
# bounds = [(0, 1)] * n_assets
# constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}]  # 가중치 합 = 1

def objective_with_soft_penalty(weights, cov, penalty=1e3, center=7.5):
    var = np.dot(weights.T, np.dot(cov, weights))
    num_selected = np.sum(weights > 0.001)  # 2% 이상인 자산만 카운트
    penalty_term = penalty * ((num_selected - center) ** 2)
    return var + penalty_term

# 무작위 초기값 설정
np.random.seed(42)
init_weights = np.random.dirichlet(np.ones(n_assets), size=1).flatten()

# 최적화 실행
res = minimize(
    fun=objective_with_soft_penalty,
    x0=init_weights,
    args=(cov,),
    method='SLSQP',
    bounds=[(0, 1)] * n_assets,
    constraints=[{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}],
    options={'disp': True, 'maxiter': 1000}
)

# 결과 정리
opt_weights = res.x
selected_assets = [asset_cols[i] for i in range(n_assets) if opt_weights[i] > 0.02]
selected_weights = [opt_weights[i] for i in range(n_assets) if opt_weights[i] > 0.02]

allocation = pd.DataFrame({'Asset': selected_assets, 'Weight': selected_weights})
allocation = allocation.sort_values(by='Weight', ascending=False)

print("[클러스터 5 - MVP 모델 결과 | 소프트 자산 수 제약 적용 (5~10개)]")
print(allocation)

### **클러스터 0 모델 다시 생각** 

#### 각 클러스터별 모델 적합한지 확인 